In [1]:
import pandas as pd

In [2]:
f=r'D:\.datasets\oil_data\train\WELL_000001\data.csv'

df = pd.read_csv(f)
df.head()

,时间,日期时间(8001),日期(8002),时间(8003),井深(8004) m,钻头位置(8005) m,迟到井深(8006) m,迟到时间(8007) min,大钩高度(8008) m,大钩速度(8009) m/s,...,硫化氢6(8221) ppm,硫化氢7(8222) ppm,硫化氢8(8223) ppm,可燃气体4(8224) ppm,可燃气体5(8225) ppm,气体流量(8226) L/s,计算全烃(8228) ppm,可燃气体1(8229) ppm,可燃气体2(8230) ppm,可燃气体3(8231) ppm
0,2025-01-07 00:00:00,-999.25,-999.25,-999.25,3875.9,3395.25,3875.9,0.0,2.59,-999.25,...,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25
1,2025-01-07 00:00:01,-999.25,-999.25,-999.25,3875.9,3395.25,3875.9,0.0,2.59,-999.25,...,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25
2,2025-01-07 00:00:04,-999.25,-999.25,-999.25,3875.9,3395.25,3875.9,0.0,2.59,-999.25,...,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25
3,2025-01-07 00:00:06,-999.25,-999.25,-999.25,3875.9,3395.25,3875.9,0.0,2.59,-999.25,...,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25
4,2025-01-07 00:00:07,-999.25,-999.25,-999.25,3875.9,3395.25,3875.9,0.0,2.59,-999.25,...,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25


In [4]:
df["timestamp"] = pd.to_datetime(
    df["时间"]
)

df = df.sort_values(
    "timestamp"
).reset_index(drop=True)

C:\Users\doyij\AppData\Local\Temp\ipykernel_38852\1866641851.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["timestamp"] = pd.to_datetime(


In [5]:
print(df["timestamp"].min()), print(df["timestamp"].max())

2025-01-07 00:00:00
2025-01-19 22:00:00


(None, None)

In [23]:
info_df = pd.read_csv(r'D:\.datasets\oil_data\train\WELL_000001\metrics.csv')
print(info_df["溢流发生时间"])

0    2025-01-17 07:30:00
Name: 溢流发生时间, dtype: str


In [8]:
overflow_time = info_df["溢流发生时间"].iloc[0]

overflow_time = pd.to_datetime(
    overflow_time
)
overflow_time

Timestamp('2025-01-17 07:30:00')

In [9]:
delta = (
    df["timestamp"]
    - overflow_time
).abs()

idx = delta.idxmin()

print(
    df.loc[idx, "timestamp"]
)

2025-01-17 07:30:00


In [12]:
WINDOW = pd.Timedelta(
    seconds=1800
)


positive_window = df[
    (df["timestamp"] >= overflow_time - WINDOW)
    &
    (df["timestamp"] <= overflow_time)
]

print(
    positive_window.shape
)

print(
    positive_window["timestamp"].min()
)

print(
    positive_window["timestamp"].max()
)

positive_window["label"]=1

(1416, 131)
2025-01-17 07:00:00
2025-01-17 07:30:00


C:\Users\doyij\AppData\Local\Temp\ipykernel_38852\1947429552.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  positive_window["label"]=1


In [15]:
positive_window.info()

<class 'pandas.DataFrame'>
RangeIndex: 1416 entries, 642188 to 643603
Columns: 132 entries, 时间 to label
dtypes: datetime64[us](1), float64(129), int64(1), str(1)
memory usage: 1.4 MB


In [21]:
import random


normal_end_candidates = df[
    df["timestamp"]
    <
    overflow_time-pd.Timedelta(minutes=60)
]["timestamp"]

end_time = random.choice(
    normal_end_candidates
)

negative_window=df[
    (df.timestamp>=end_time-WINDOW)
    &
    (df.timestamp<=end_time)
]

negative_window["label"]=0

C:\Users\doyij\AppData\Local\Temp\ipykernel_38852\2132032722.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  negative_window["label"]=0


In [22]:
negative_window.info()

<class 'pandas.DataFrame'>
RangeIndex: 1448 entries, 459990 to 461437
Columns: 132 entries, 时间 to label
dtypes: datetime64[us](1), float64(129), int64(1), str(1)
memory usage: 1.5 MB


In [26]:
df.shape

(807670, 131)